## Case Demanding:

- Data Review and Storage:
Review the loaded data and assign appropriate data types based on your best judgment.
Identify primary and foreign keys for each table.
Store the transformed data using a store_ prefix.

#### Environment Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

#### Products CSV Review and Merge

In [0]:
try:
  # Getting just files not loaded on the store table
  df_products = spark.table('interviewcaseupstart.sales_db.raw_products').filter(F.col('file_loaded_store') == False)

  # Casting the collumns to apropriate types
  df_products_casted = (
    df_products 
    .withColumn('ProductID',F.col('ProductID').cast(IntegerType()))
    .withColumn('ProductDesc',F.col('ProductDesc').cast(StringType()))
    .withColumn('ProductNumber',F.col('ProductNumber').cast(StringType()))
    .withColumn('MakeFlag',F.col('MakeFlag').cast(BooleanType()))
    .withColumn('Color',F.col('Color').cast(StringType()))
    .withColumn('SafetyStockLevel',F.col('SafetyStockLevel').cast(IntegerType()))
    .withColumn('ReorderPoint',F.col('ReorderPoint').cast(IntegerType()))
    .withColumn('StandardCost',F.col('StandardCost').cast(DecimalType(10, 4)))
    .withColumn('ListPrice',F.col('ListPrice').cast(DecimalType(10, 3)))
    .withColumn('Size',F.col('Size').cast(StringType()))
    .withColumn('SizeUnitMeasureCode',F.col('SizeUnitMeasureCode').cast(StringType()))
    .withColumn('Weight',F.col('Weight').cast(DecimalType(10, 2)))
    .withColumn('WeightUnitMeasureCode',F.col('WeightUnitMeasureCode').cast(StringType()))
    .withColumn('ProductCategoryName',F.col('ProductCategoryName').cast(StringType()))
    .withColumn('ProductSubcategoryName',F.col('ProductSubcategoryName').cast(StringType()))
    .withColumn('LastModificationTime',F.lit(datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")).cast(TimestampType()))
    .withColumn('ProductPublished',F.lit(False))
  )

  # Droping the duplicates based on ProductCategoryName for each ProductID and droping raw collumns
  df_new_products = df_products_casted.drop('file_modification_time','file_path','file_name','file_size','file_loading_time','file_loaded_store')
  df_new_products = df_new_products.withColumn('row_number',F.row_number().over(Window.partitionBy('ProductID').orderBy(F.desc('ProductCategoryName'))))
  df_new_products = df_new_products.filter(F.col('row_number') == 1)
  df_new_products = df_new_products.drop('row_number')

  # Getting file_loaded_store flag to be updated
  df_new_products_bronze_update = df_products_casted.select('file_modification_time','file_path','file_size','file_loading_time').drop_duplicates()
  df_new_products_bronze_update = df_new_products_bronze_update.withColumn('file_loaded_store',F.lit(True))
except Exception as e:
  dbutils.notebook.exit(e)

In [0]:
try:
  delta_table_silver = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.store_products')

  (delta_table_silver.alias("T")
    .merge(
      df_new_products.alias("S"),
      "S.ProductID = T.ProductID")
    .whenMatchedUpdate(set =
      {
        "ProductDesc": "S.ProductDesc",
        "ProductNumber": "S.ProductNumber",
        "MakeFlag": "S.MakeFlag",
        "Color": "S.Color",
        "SafetyStockLevel": "S.SafetyStockLevel",
        "ReorderPoint": "S.ReorderPoint",
        "StandardCost": "S.StandardCost",
        "ListPrice": "S.ListPrice",
        "Size": "S.Size",
        "SizeUnitMeasureCode": "S.SizeUnitMeasureCode",
        "WeightUnitMeasureCode": "S.WeightUnitMeasureCode",
        "Weight": "S.Weight",
        "ProductCategoryName": "S.ProductCategoryName",
        "ProductSubcategoryName": "S.ProductSubcategoryName",
        "LastModificationTime": "S.LastModificationTime",
        "ProductPublished": "S.ProductPublished"
      })
    .whenNotMatchedInsertAll()
    .execute()
  )

  delta_table_bronze = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.raw_products')

  (delta_table_bronze.alias("T")
    .merge(
      df_new_products_bronze_update.alias("S"),
      "T.file_modification_time = S.file_modification_time and T.file_path = S.file_path and T.file_loading_time = S.file_loading_time and T.file_loaded_store = false")
    .whenMatchedUpdate(set =
      {
        "file_loaded_store": "S.file_loaded_store"
      })
    .execute()
  )
except Exception as e:
  dbutils.notebook.exit(e)

#### Order Detail CSV Review and Merge

In [0]:
try:
  # Getting just files not loaded on the store table
  df_sales_order_detail = spark.table('interviewcaseupstart.sales_db.raw_sales_order_detail').filter(F.col('file_loaded_store') == False)

  # Casting the collumns to apropriate types
  df_sales_order_detail_casted = (
    df_sales_order_detail 
    .withColumn('SalesOrderID',F.col('SalesOrderID').cast(IntegerType()))
    .withColumn('SalesOrderDetailID',F.col('SalesOrderDetailID').cast(IntegerType()))
    .withColumn('OrderQty',F.col('OrderQty').cast(IntegerType()))
    .withColumn('ProductID',F.col('ProductID').cast(IntegerType()))
    .withColumn('UnitPrice',F.col('UnitPrice').cast(DecimalType(10, 4)))
    .withColumn('UnitPriceDiscount',F.col('UnitPriceDiscount').cast(DecimalType(10, 4)))
    .withColumn('LastModificationTime',F.lit(datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")).cast(TimestampType()))
    .withColumn('OrderDetailPublished',F.lit(False))
  )

  # Droping droping raw collumns
  df_new_sales_order_detail = df_sales_order_detail_casted.drop('file_modification_time','file_path','file_name','file_size','file_loading_time','file_loaded_store')

  # Getting file_loaded_store flag to be updated
  df_new_sales_order_detail_bronze_update = df_sales_order_detail_casted.select('file_modification_time','file_path','file_size','file_loading_time').drop_duplicates()
  df_new_sales_order_detail_bronze_update = df_new_sales_order_detail_bronze_update.withColumn('file_loaded_store',F.lit(True))
except Exception as e:
  dbutils.notebook.exit(e)

In [0]:
try:
  delta_table_silver_order_detail = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.store_sales_order_detail')

  (delta_table_silver_order_detail.alias("T")
    .merge(
      df_new_sales_order_detail.alias("S"),
      "S.ProductID = T.ProductID and S.SalesOrderID = T.SalesOrderID and S.SalesOrderDetailID = T.SalesOrderDetailID")
    .whenMatchedUpdate(set =
      {
        "OrderQty": "S.OrderQty",
        "UnitPrice": "S.UnitPrice",
        "UnitPriceDiscount": "S.UnitPriceDiscount",
        "LastModificationTime": "S.LastModificationTime",
        "OrderDetailPublished": "S.OrderDetailPublished"
      })
    .whenNotMatchedInsertAll()
    .execute()
  )

  delta_table_bronze_order_detail = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.raw_sales_order_detail')

  (delta_table_bronze_order_detail.alias("T")
    .merge(
      df_new_sales_order_detail_bronze_update.alias("S"),
      "T.file_modification_time = S.file_modification_time and T.file_path = S.file_path and T.file_loading_time = S.file_loading_time and T.file_loaded_store = false")
    .whenMatchedUpdate(set =
      {
        "file_loaded_store": "S.file_loaded_store"
      })
    .execute()
  )
except Exception as e:
  dbutils.notebook.exit(e)

#### Order Header CSV Review and Merge

In [0]:
try:
  # Getting just files not loaded on the store table
  df_sales_order_header = spark.table('interviewcaseupstart.sales_db.raw_sales_order_header').filter(F.col('file_loaded_store') == False)

  # Casting the collumns to apropriate types, note that null dates are not well formated dates
  df_sales_order_header_casted = (
    df_sales_order_header
    .withColumn('SalesOrderID',F.col('SalesOrderID').cast(IntegerType()))
    .withColumn('OrderDate',F.try_to_date("OrderDate", "yyyy-MM-dd"))
    .withColumn('ShipDate',F.try_to_date("ShipDate", "yyyy-MM-dd"))
    .withColumn('OnlineOrderFlag',F.col('OnlineOrderFlag').cast(BooleanType()))
    .withColumn('AccountNumber',F.col('AccountNumber').cast(StringType()))
    .withColumn('CustomerID',F.col('CustomerID').cast(IntegerType()))
    .withColumn('SalesPersonID',F.col('SalesPersonID').cast(IntegerType()))
    .withColumn('Freight',F.col('Freight').cast(DecimalType(10, 4)))
    .withColumn('LastModificationTime',F.lit(datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")).cast(TimestampType()))
    .withColumn('OrderHeaderPublished',F.lit(False))
  )

  # Droping droping raw collumns
  df_new_sales_order_header = df_sales_order_header_casted.drop('file_modification_time','file_path','file_name','file_size','file_loading_time','file_loaded_store')

  # Getting file_loaded_store flag to be updated
  df_new_sales_order_header_bronze_update = df_sales_order_header_casted.select('file_modification_time','file_path','file_size','file_loading_time').drop_duplicates()
  df_new_sales_order_header_bronze_update = df_new_sales_order_header_bronze_update.withColumn('file_loaded_store',F.lit(True))
except Exception as e:
  dbutils.notebook.exit(e)

In [0]:
try:
  delta_table_silver_order_header = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.store_sales_order_header')

  (delta_table_silver_order_header.alias("T")
    .merge(
      df_new_sales_order_header.alias("S"),
      "S.SalesOrderID = T.SalesOrderID and S.CustomerID = T.CustomerID")
    .whenMatchedUpdate(set =
      {
        "OrderDate": "S.OrderDate",
        "ShipDate": "S.ShipDate",
        "OnlineOrderFlag": "S.OnlineOrderFlag",
        "AccountNumber": "S.AccountNumber",
        "SalesPersonID": "S.SalesPersonID",
        "Freight": "S.Freight",
        "LastModificationTime": "S.LastModificationTime",
        "OrderHeaderPublished": "S.OrderHeaderPublished"
      })
    .whenNotMatchedInsertAll()
    .execute()
  )

  delta_table_bronze_order_header = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.raw_sales_order_header')

  (delta_table_bronze_order_header.alias("T")
    .merge(
      df_new_sales_order_header_bronze_update.alias("S"),
      "T.file_modification_time = S.file_modification_time and T.file_path = S.file_path and T.file_loading_time = S.file_loading_time and T.file_loaded_store = false")
    .whenMatchedUpdate(set =
      {
        "file_loaded_store": "S.file_loaded_store"
      })
    .execute()
  )
except Exception as e:
  dbutils.notebook.exit(e)